# Bengali Hallucination Dataset — WH Question/Answer Generation (Qwen3.5, GPU)

Generates 8 WH question-answer pairs per context using **Qwen3.5** via `transformers` + 4-bit quantization, batched on GPU. Replaces the local-Ollama version with direct GPU inference for speed.

**Before running:**
- Kaggle notebook settings → Accelerator → **GPU T4 x2**
- Settings → Internet → **On** (needed to download the model from Hugging Face)
- Attach the `bengali-hallucination` competition dataset

**Model sizing for T4x2 (16GB each):**
| Model | VRAM (4-bit) | Notes |
|---|---|---|
| `Qwen/Qwen3.5-4B` | ~3GB | fastest, lowest quality |
| `Qwen/Qwen3.5-9B` | ~6GB | **recommended default** — fits on 1 T4 with headroom to batch |
| `Qwen/Qwen3.5-35B-A3B` (MoE) | ~18-20GB | needs both T4s via `device_map="auto"`, slower setup, better quality |


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import json, re, os, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("GPUs available:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  {i}: {torch.cuda.get_device_name(i)} - {props.total_memory/1e9:.1f} GB")

## Config

In [ ]:
# ---- Paths ----
DATA_DIR = "/kaggle/input/competitions/bengali-hallucination"
INPUT_PATH = f"{DATA_DIR}/dataset samples.json"
OUTPUT_PATH = "/kaggle/working/question_answer.json"
PROGRESS_PATH = "/kaggle/working/question_answer.progress.json"
CHECKPOINT_EVERY = 20     # save progress every N contexts processed

# ---- Model ----
# Swap this to Qwen/Qwen3.5-4B for more speed, or Qwen/Qwen3.5-35B-A3B for more quality (needs both GPUs)
MODEL_NAME = "Qwen/Qwen3.5-9B"

# ---- Generation ----
BATCH_SIZE = 8            # contexts processed per generate() call - raise if VRAM allows
MAX_NEW_TOKENS = 700
LIMIT = None               # set an int (e.g. 20) for a quick smoke test before a full run

## Load model (4-bit quantized)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"          # required for correct batched generation on decoder-only models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()

print("Model loaded. Device map:", getattr(model, "hf_device_map", "n/a"))

## Prompt template + parsing helpers
(ported directly from `generate_qa.py`)

In [ ]:
NULL_MARKERS = {"[NULL]", "", None}

PROMPT_TEMPLATE = """তুমি একজন বাংলা ভাষা বিশেষজ্ঞ। নিচের প্যারাগ্রাফটি পড়ো এবং শুধুমাত্র এই প্যারাগ্রাফে উল্লেখিত তথ্যের ভিত্তিতে ৮টি "প্রশ্ন-উত্তর" জোড়া তৈরি করো।

নিয়মাবলি:
1. প্রতিটি প্রশ্ন হতে হবে WH (কী/কে/কবে/কোথায়/কেন/কীভাবে/কোনটি/কতজন ইত্যাদি) ধরনের।
2. প্রতিটি উত্তর অবশ্যই প্যারাগ্রাফে সরাসরি উল্লেখিত তথ্য থেকে আসতে হবে, নিজে থেকে কোনো তথ্য যোগ করবে না।
3. উত্তর যতটা সম্ভব সংক্ষিপ্ত রাখো (একটি শব্দ বা বাক্যাংশ), সম্পূর্ণ বাক্য নয়।
4. একই তথ্য নিয়ে দুইবার প্রশ্ন করো না, প্যারাগ্রাফের বিভিন্ন অংশ কভার করো।
5. আউটপুট অবশ্যই শুধুমাত্র বৈধ JSON array হতে হবে, অন্য কোনো ব্যাখ্যা বা টেক্সট থাকবে না।

আউটপুট ফরম্যাট (উদাহরণ):
[
  {{"question": "...", "answer": "..."}},
  ...
]

প্যারাগ্রাফ:
\"\"\"{context}\"\"\"
"""

def build_prompt(context):
    return PROMPT_TEMPLATE.format(context=context)

def extract_json_array(text):
    """Pull out the first JSON array found in the model output, tolerating
    stray preamble/markdown fences despite instructions not to add them."""
    text = text.strip()
    text = re.sub(r"^```(json)?", "", text)
    text = re.sub(r"```$", "", text)
    text = text.strip()

    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1 or end < start:
        return None

    candidate = text[start:end + 1]
    try:
        data = json.loads(candidate)
        if isinstance(data, list):
            return data
    except json.JSONDecodeError:
        return None
    return None

def normalize(s):
    return re.sub(r"[\s,।.()\"\'\u200c\u200d]+", "", s or "")

def is_grounded(answer, context):
    return normalize(answer) in normalize(context)

## Batched generation
This is the actual speedup over the Ollama version: several contexts go through `model.generate()` in one GPU call instead of one HTTP request at a time.

In [ ]:
@torch.inference_mode()
def generate_batch(contexts):
    """Returns a list of raw decoded model outputs, one per context."""
    messages_list = [[{"role": "user", "content": build_prompt(c)}] for c in contexts]

    try:
        prompts = [
            tokenizer.apply_chat_template(
                m, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            for m in messages_list
        ]
    except TypeError:
        # fallback if this tokenizer's chat template doesn't accept enable_thinking
        prompts = [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in messages_list
        ]

    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048
    ).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

    input_len = inputs["input_ids"].shape[1]
    generated = output_ids[:, input_len:]
    return tokenizer.batch_decode(generated, skip_special_tokens=True)

## Load dataset

In [ ]:
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    samples = json.load(f)

if LIMIT:
    samples = samples[:LIMIT]

contexts = [s.get("context") for s in samples if s.get("context") not in NULL_MARKERS]
print(f"Total contexts to process: {len(contexts)}")

## Main loop (with checkpointing/resume)
Saves progress every `CHECKPOINT_EVERY` contexts, so a disconnected Kaggle session can pick back up instead of losing the run.

In [ ]:
all_pairs = []
start_idx = 0

if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        prog = json.load(f)
    all_pairs = prog["pairs"]
    start_idx = prog["next_index"]
    print(f"Resuming from context {start_idx}, {len(all_pairs)} pairs already collected")

t0 = time.time()
for batch_start in range(start_idx, len(contexts), BATCH_SIZE):
    batch = contexts[batch_start: batch_start + BATCH_SIZE]
    raw_outputs = generate_batch(batch)

    for raw in raw_outputs:
        qa_pairs = extract_json_array(raw)
        if not qa_pairs:
            print(f"  -> FAILED to parse output at batch starting {batch_start}")
            continue
        for qa in qa_pairs[:8]:
            q = (qa.get("question") or "").strip()
            a = (qa.get("answer") or "").strip()
            if q and a:
                all_pairs.append({"Question": q, "Answer": a})

    done = batch_start + len(batch)
    elapsed = time.time() - t0
    rate = done / elapsed if elapsed > 0 else 0
    print(f"[{done}/{len(contexts)}] pairs so far: {len(all_pairs)}  "
          f"({elapsed:.0f}s elapsed, {rate:.2f} contexts/s)")

    if done % CHECKPOINT_EVERY == 0 or done == len(contexts):
        with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
            json.dump({"pairs": all_pairs, "next_index": done}, f, ensure_ascii=False)

## Save final output

In [ ]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_pairs, f, ensure_ascii=False, indent=2)

print(f"Done. Wrote {OUTPUT_PATH}")
print(f"Total Q&A pairs generated: {len(all_pairs)}")

## Sanity check

In [ ]:
sample_check = all_pairs[:5]
for pair in sample_check:
    print(pair)

grounded = sum(1 for p in all_pairs if is_grounded(p["Answer"], " ".join(contexts)))
print(f"\nRoughly grounded (loose check across all contexts): {grounded}/{len(all_pairs)}")